# cap sweep: asym vs pipeline 同机同码对照(R=1024, N=10)

阶段0 补充实验。上一轮 16-trial(`asym_notebook_trials.csv`)只测了 asym 单侧、
每配置 1 次 repeat、且"零跨卡"的两条论据都不严谨(cap 不敏感在 pipeline 基线同样成立;
nvidia-smi 快照拍在 sweep 结束之后)。本 notebook 补齐三件事:

1. **pipeline 对照组**——同机、同代码、同 seed 实测 13/15 accelerate 流水线, 替代旧 sweep 的 18.2 tok/s 间接基线;
2. **think 进行中 GPU 利用率采样**——后台线程 0.2s 采一次 nvidia-smi, asym 模式 gen 卡应 ≈0, 这是"零跨卡"的直接证据;
3. **每配置 3 trials** 出均值±标准差, 以及固定 seed 下两种放置的输出文本一致性比对。

| 维度 | 取值 |
|------|------|
| placement | asym → pipeline(同 kernel 内先后跑, 中间释放显存) |
| R | 1024(固定) |
| num_timesteps | 10(固定) |
| cap | 128 / 256 / 512 / 1000(budget forcing, min=max=cap) |
| trials | 3 × 4 prompts = 48 / placement, 共 96 |

预计耗时: asym sweep ~25 min + pipeline sweep ~27 min + 两次加载 ~10 min ≈ **1 小时**。

两个 sweep cell 均有断点续跑保护: 输出 CSV 已存在时自动跳过(`FORCE_RERUN=True` 强制重跑)。
若 asym 跑完后 pipeline 加载因显存碎片 OOM, Restart Kernel 从头 Run All 即可——asym 段会
因 CSV 已存在而秒过, 直接进入 pipeline 段。


## 1. 环境 / import

`CUDA_VISIBLE_DEVICES` 必须在 import torch 之前设置(torch 首次 import 锁定可见设备,
`docs/DEBUG_NOTES.md §8.4`)。优先用 kernel.json env 注入; 缺失则 setdefault 兜底,
若本 kernel 已 import 过 torch 需 Restart Kernel。


In [ ]:
import os
import sys
import time
import gc
import threading
import subprocess
from copy import deepcopy
from contextlib import nullcontext

# ── CUDA_VISIBLE_DEVICES: 必须在 import torch 之前 (DEBUG_NOTES.md §8.4) ──
_pre_cvd = os.environ.get("CUDA_VISIBLE_DEVICES")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
if _pre_cvd is None:
    print(f"⚠ CUDA_VISIBLE_DEVICES 未在 kernel 启动时注入, setdefault 兜底为 "
          f"{os.environ['CUDA_VISIBLE_DEVICES']} — 若本 kernel 已 import 过 torch 请 Restart Kernel")
else:
    print(f"CUDA_VISIBLE_DEVICES = {_pre_cvd} (kernel.json env 注入)")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# 允许从 repo root / experiments/ / experiments/notebooks/ 启动
_cwd = os.getcwd()
if os.path.basename(_cwd) == "notebooks":
    _proj_root = os.path.abspath(os.path.join(_cwd, "..", ".."))
elif os.path.basename(_cwd) == "experiments":
    _proj_root = os.path.abspath(os.path.join(_cwd, ".."))
else:
    _proj_root = _cwd
if _proj_root not in sys.path:
    sys.path.insert(0, _proj_root)
os.chdir(_proj_root)
print(f"project root: {_proj_root}")

import numpy as np
import torch
import pandas as pd

print(f"torch {torch.__version__}, visible GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")
assert torch.cuda.device_count() >= 2, "asym / pipeline 双卡实验需要 2 张 GPU 可见"


## 2. 实验配置 + 复用工具

直接复用 `experiments/scripts/run_asym_bench.py` 的 trial 结构 / 计时器 / 条件构造 /
pipeline 加载器。`run_trial_ext` 是 `run_trial` 的复制版, **唯一改动**是 think 阶段
(`gen_text` 调用)套一层 `GpuUtilSampler` 后台采样两张物理卡的利用率。

seed 公式与 `run_asym_bench.py` 完全一致(`SEED_BASE + pi*10000 + ci*100 + repeat`),
因此 asym / pipeline 逐 trial 同 seed, 可做输出一致性比对。


In [ ]:
from experiments.scripts.asym_placement import (
    load_model_asym, verify_placement, install_input_transfer_shim,
)
from experiments.scripts.run_asym_bench import (
    _load_model_pipeline, sync_timer, set_all_seeds, reset_taylorseer_state,
    build_conditions, IMAGE_SHAPE, BENCH_PROMPTS, SEED_BASE, WARMUP_SEED,
    BASELINE_T_THINK_PER_TOKEN, TARGET_T_THINK_TOK_PER_SEC,
)
from data.transforms import ImageTransform
from inferencer import InterleaveInferencer, GEN_THINK_SYSTEM_PROMPT

MODEL_PATH = os.path.join(_proj_root, "BAGEL-7B-MoT")
OUTPUT_DIR = os.path.join(_proj_root, "experiments", "outputs", "asym_bench_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CAPS = [128, 256, 512, 1000]
NUM_TIMESTEPS = [10]
N_TRIALS = 3
PROMPTS = BENCH_PROMPTS  # 4 条, 与上一轮 16-trial 相同
CONDITIONS = build_conditions(CAPS, NUM_TIMESTEPS)  # R=IMAGE_SHAPE=1024², N=10, 4 caps

ASYM_CSV = os.path.join(OUTPUT_DIR, "cap_sweep_asym_trials.csv")
PIPELINE_CSV = os.path.join(OUTPUT_DIR, "cap_sweep_pipeline_trials.csv")

FORCE_RERUN = False
RUN_ASYM = FORCE_RERUN or not os.path.exists(ASYM_CSV)
RUN_PIPELINE = FORCE_RERUN or not os.path.exists(PIPELINE_CSV)
print(f"conditions={len(CONDITIONS)} (caps={CAPS}, ts={NUM_TIMESTEPS}), "
      f"trials/cfg={N_TRIALS}, prompts={len(PROMPTS)}")
print(f"RUN_ASYM={RUN_ASYM}, RUN_PIPELINE={RUN_PIPELINE} (FORCE_RERUN={FORCE_RERUN})")


In [ ]:
class GpuUtilSampler:
    """后台线程按物理卡号采样 nvidia-smi 利用率, 用 with 包住 think 阶段。

    nvidia-smi 的 utilization.gpu 本身是 ≤1s 窗口的滑动值, 对 5~40s 的 think
    段足够; cap=128 (~5s) 也有 ~25 个样本。采样失败(无 nvidia-smi 等)静默跳过,
    stats() 返回 n=0。
    """

    def __init__(self, interval=0.2):
        cvd = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1")
        self.phys = [x.strip() for x in cvd.split(",") if x.strip()][:2]
        self.interval = interval
        self._stop = threading.Event()
        self._thread = None
        self.samples = []

    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=index,utilization.gpu",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5,
                ).stdout
                util = {}
                for line in out.strip().splitlines():
                    idx, u = [t.strip() for t in line.split(",")]
                    util[idx] = float(u)
                self.samples.append(tuple(util.get(p, float("nan")) for p in self.phys))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        self.samples = []
        self._stop.clear()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join(timeout=2)

    def stats(self):
        if not self.samples:
            return dict(n=0)
        arr = np.array(self.samples, dtype=float)
        return dict(
            n=len(arr),
            gpu0_mean=float(np.nanmean(arr[:, 0])), gpu0_max=float(np.nanmax(arr[:, 0])),
            gpu1_mean=float(np.nanmean(arr[:, 1])), gpu1_max=float(np.nanmax(arr[:, 1])),
        )


def run_trial_ext(inferencer, tokenizer, prompt, cond, seed, sample_gpu=True):
    """复制自 run_asym_bench.run_trial, 唯一改动: think 阶段套 GpuUtilSampler。"""
    reset_taylorseer_state(inferencer.model)
    set_all_seeds(seed)
    record = dict(prompt=prompt, seed=seed, **cond)
    sampler = GpuUtilSampler() if sample_gpu else None
    gen_context = cfg_text_context = cfg_img_context = None
    try:
        gen_context = inferencer.init_gen_context()
        cfg_text_context = deepcopy(gen_context)
        cfg_img_context = deepcopy(gen_context)

        with torch.autocast(device_type="cuda", enabled=True, dtype=torch.bfloat16):
            with sync_timer() as t_prefill:
                gen_context = inferencer.update_context_text(GEN_THINK_SYSTEM_PROMPT, gen_context)
                cfg_img_context = inferencer.update_context_text(GEN_THINK_SYSTEM_PROMPT, cfg_img_context)
                cfg_text_context = deepcopy(gen_context)
                gen_context = inferencer.update_context_text(prompt, gen_context)
                cfg_img_context = inferencer.update_context_text(prompt, cfg_img_context)

            with sync_timer() as t_think:
                with (sampler or nullcontext()):
                    gen_text = inferencer.gen_text(
                        gen_context, do_sample=False, temperature=0.3,
                        max_length=cond["max_think_token_n"],
                        min_length=cond.get("min_think_token_n", 0),
                        wait_interjection=cond.get("wait_interjection"),
                    )

            gen_context = inferencer.update_context_text(gen_text, gen_context)

            with sync_timer() as t_image:
                inferencer.gen_image(
                    IMAGE_SHAPE, gen_context,
                    cfg_text_precontext=cfg_text_context,
                    cfg_img_precontext=cfg_img_context,
                    cfg_text_scale=cond["cfg_text_scale"],
                    cfg_img_scale=cond["cfg_img_scale"],
                    cfg_interval=cond["cfg_interval"],
                    cfg_renorm_min=cond["cfg_renorm_min"],
                    cfg_renorm_type=cond["cfg_renorm_type"],
                    timestep_shift=cond["timestep_shift"],
                    num_timesteps=cond["num_timesteps"],
                    enable_taylorseer=cond["enable_taylorseer"],
                )

        think_token_count = len(tokenizer(gen_text, add_special_tokens=False).input_ids)
        record.update(
            t_prefill=t_prefill.elapsed, t_think=t_think.elapsed, t_image=t_image.elapsed,
            think_token_count=think_token_count,
            think_tok_per_sec=think_token_count / t_think.elapsed if t_think.elapsed > 0 else None,
            hit_cap=think_token_count >= cond["max_think_token_n"] - 2,
            think_closed=gen_text.strip().endswith("</think>"),
            gen_text=gen_text, ok=True, error=None,
        )
        if sampler is not None:
            record.update({f"think_{k}": v for k, v in sampler.stats().items()})
    except Exception as e:
        record.update(
            t_prefill=None, t_think=None, t_image=None,
            think_token_count=None, think_tok_per_sec=None,
            hit_cap=None, think_closed=None, gen_text=None, ok=False, error=repr(e),
        )
    finally:
        del gen_context, cfg_text_context, cfg_img_context
        reset_taylorseer_state(inferencer.model)
        gc.collect()
        torch.cuda.empty_cache()
    return record


def run_sweep(inferencer, tokenizer, placement, out_csv):
    """warmup 1 次 + 4 prompts × 4 caps × 3 trials, 写 CSV, 返回 DataFrame。"""
    w_cond = CONDITIONS[0]
    print(f"[{placement}] warmup: cap={w_cond['max_think_token_n']} ts={w_cond['num_timesteps']} ...")
    w = run_trial_ext(inferencer, tokenizer, "a photo of a cat", w_cond,
                      seed=WARMUP_SEED, sample_gpu=False)
    assert w["ok"], f"[{placement}] warmup FAILED: {w['error']}"
    print(f"[{placement}] warmup ok: t_think={w['t_think']:.2f}s "
          f"t_image={w['t_image']:.2f}s tok/s={w['think_tok_per_sec']:.1f}")

    rows, t0, i = [], time.perf_counter(), 0
    total = len(PROMPTS) * len(CONDITIONS) * N_TRIALS
    for pi, prompt in enumerate(PROMPTS):
        for ci, cond in enumerate(CONDITIONS):
            for repeat in range(N_TRIALS):
                i += 1
                seed = SEED_BASE + pi * 10000 + ci * 100 + repeat
                row = run_trial_ext(inferencer, tokenizer, prompt, cond, seed)
                row.update(placement=placement, prompt_idx=pi, cond_idx=ci,
                           repeat=repeat, trial_idx=i)
                rows.append(row)
                el = time.perf_counter() - t0
                eta = el / i * (total - i)
                st = "ok" if row["ok"] else f"FAIL({str(row['error'])[:60]})"
                tps = row.get("think_tok_per_sec")
                g1 = row.get("think_gpu1_mean")
                print(f"[{placement}] [{i}/{total}] cap={cond['max_think_token_n']} rep={repeat}: {st} "
                      f"tok/s={tps and round(tps, 2)} "
                      f"t_image={row.get('t_image') and round(row['t_image'], 1)}s "
                      f"gpu1_util(think)={g1 if g1 is None else round(g1, 1)}% "
                      f"elapsed={el:.0f}s ETA={eta:.0f}s")

    df = pd.DataFrame(rows)
    df["hardware"] = torch.cuda.get_device_name(0)
    df["timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%S")
    df.to_csv(out_csv, index=False)
    print(f"[{placement}] wrote {len(df)} rows → {out_csv}")
    return df


## 3. 加载 asym 模型(+ verify + shim)

`load_model_asym` → `verify_placement(strict=True)` → `install_input_transfer_shim`
(P0 修复, 6 个 `prepare_*` 必须全部装上)。CSV 已存在时整段跳过, 不浪费加载时间。


In [ ]:
if RUN_ASYM:
    _t0 = time.perf_counter()
    model, vae_model, tokenizer, new_token_ids = load_model_asym(
        model_path=MODEL_PATH, und_device="cuda:0", gen_device="cuda:1",
    )
    verify_placement(model, und_device="cuda:0", gen_device="cuda:1",
                     vae_model=vae_model, strict=True)
    _n = install_input_transfer_shim(model, und_device="cuda:0")
    assert _n == 6, f"expected 6 shimmed prepare_* methods, got {_n}"

    vae_transform = ImageTransform(1024, 512, 16)
    vit_transform = ImageTransform(980, 224, 14)
    inferencer = InterleaveInferencer(
        model=model, vae_model=vae_model, tokenizer=tokenizer,
        vae_transform=vae_transform, vit_transform=vit_transform,
        new_token_ids=new_token_ids,
    )
    print(f"[asym] ready in {time.perf_counter() - _t0:.0f}s")
else:
    print(f"跳过 asym 加载({ASYM_CSV} 已存在, FORCE_RERUN=False)")


## 4. asym sweep(48 trials, 预计 ~25 min)

进度行里的 `gpu1_util(think)` 是 think 阶段 gen 卡的平均利用率——**应 ≈0**。
如果明显 >0, 说明 think 路径仍有跨卡, 停下来查。


In [ ]:
if RUN_ASYM:
    df_asym = run_sweep(inferencer, tokenizer, "asym", ASYM_CSV)
else:
    df_asym = pd.read_csv(ASYM_CSV)
    print(f"从 {ASYM_CSV} 载入 {len(df_asym)} 行")

_ok = df_asym[df_asym["ok"] == True]
print(_ok.groupby("max_think_token_n")["think_tok_per_sec"]
         .agg(["mean", "std", "count"]).round(2))


## 5. 释放 asym, 加载 pipeline 对照组

同 kernel 内先 `del` 掉 asym 的模型/推理器, 两张卡分别 `empty_cache`, 再走
`_load_model_pipeline`(13/15 accelerate 加载, 与 `run_cap_sweep_mp.py:272-356` 一致)。
若此处 OOM(显存碎片), Restart Kernel → Run All: asym 段会因 CSV 已存在秒过。


In [ ]:
if RUN_PIPELINE:
    for _v in ("inferencer", "model", "vae_model", "tokenizer", "new_token_ids"):
        if _v in globals():
            del globals()[_v]
    gc.collect()
    for _i in range(torch.cuda.device_count()):
        with torch.cuda.device(_i):
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        _free, _tot = torch.cuda.mem_get_info(_i)
        print(f"cuda:{_i} free {_free / 1024**3:.1f} / {_tot / 1024**3:.1f} GiB")

    _t0 = time.perf_counter()
    model, vae_model, tokenizer, new_token_ids = _load_model_pipeline(MODEL_PATH)
    vae_transform = ImageTransform(1024, 512, 16)
    vit_transform = ImageTransform(980, 224, 14)
    inferencer = InterleaveInferencer(
        model=model, vae_model=vae_model, tokenizer=tokenizer,
        vae_transform=vae_transform, vit_transform=vit_transform,
        new_token_ids=new_token_ids,
    )
    print(f"[pipeline] ready in {time.perf_counter() - _t0:.0f}s")
else:
    print(f"跳过 pipeline 加载({PIPELINE_CSV} 已存在, FORCE_RERUN=False)")


## 6. pipeline sweep(48 trials, 预计 ~27 min)

pipeline 模式 think 阶段两张卡都应有利用率(前 13 层 GPU0, 后 15 层 GPU1),
`gpu1_util(think)` 显著 >0 属预期——与 asym 的 ≈0 形成对照。


In [ ]:
if RUN_PIPELINE:
    df_pipe = run_sweep(inferencer, tokenizer, "pipeline", PIPELINE_CSV)
else:
    df_pipe = pd.read_csv(PIPELINE_CSV)
    print(f"从 {PIPELINE_CSV} 载入 {len(df_pipe)} 行")

_ok = df_pipe[df_pipe["ok"] == True]
print(_ok.groupby("max_think_token_n")["think_tok_per_sec"]
         .agg(["mean", "std", "count"]).round(2))


## 7. 汇总: 每 cap 的 tok/s、加速比、t_image 回归


In [ ]:
df = pd.concat([pd.read_csv(ASYM_CSV), pd.read_csv(PIPELINE_CSV)], ignore_index=True)
ok = df[df["ok"] == True].copy()
print(f"trials: {len(df)} total, {len(ok)} ok, {len(df) - len(ok)} failed")
if len(df) - len(ok):
    print(df[df["ok"] != True][["placement", "max_think_token_n", "repeat", "error"]])

agg = ok.groupby(["placement", "max_think_token_n"]).agg(
    tok_per_sec_mean=("think_tok_per_sec", "mean"),
    tok_per_sec_std=("think_tok_per_sec", "std"),
    t_think_mean=("t_think", "mean"),
    t_image_mean=("t_image", "mean"),
    t_prefill_ms=("t_prefill", lambda s: s.mean() * 1000),
    n=("ok", "count"),
).round(3)
print(agg.to_string())

piv_tps = ok.pivot_table(index="max_think_token_n", columns="placement",
                         values="think_tok_per_sec", aggfunc="mean")
piv_tps["speedup"] = (piv_tps["asym"] / piv_tps["pipeline"]).round(3)
piv_img = ok.pivot_table(index="max_think_token_n", columns="placement",
                         values="t_image", aggfunc="mean")
piv_img["t_image_ratio"] = (piv_img["asym"] / piv_img["pipeline"]).round(3)
print()
print("think tok/s(asym vs pipeline):")
print(piv_tps.round(2).to_string())
print()
print("t_image(asym vs pipeline, N=10):")
print(piv_img.round(2).to_string())

_mean_asym = ok[ok["placement"] == "asym"]["think_tok_per_sec"].mean()
_mean_pipe = ok[ok["placement"] == "pipeline"]["think_tok_per_sec"].mean()
print()
print(f"overall: asym {_mean_asym:.2f} tok/s vs pipeline {_mean_pipe:.2f} tok/s "
      f"(x{_mean_asym / _mean_pipe:.2f}); 旧间接基线 {1 / BASELINE_T_THINK_PER_TOKEN:.1f} tok/s")
if _mean_asym >= TARGET_T_THINK_TOK_PER_SEC:
    print(f"✅ 达到 ≥{TARGET_T_THINK_TOK_PER_SEC} tok/s 目标")
else:
    print(f"❌ 未达 ≥{TARGET_T_THINK_TOK_PER_SEC} tok/s 目标(差 "
          f"{(1 - _mean_asym / TARGET_T_THINK_TOK_PER_SEC) * 100:.0f}%)")


## 8. "零跨卡"直接验证: think 阶段 GPU 利用率

上一轮的两条论据都不严谨(cap 不敏感在 pipeline 基线同样成立; nvidia-smi 快照拍在
sweep 结束后)。这里用 think 阶段**进行中**的采样直接验证:

- **asym**: `think_gpu1_mean ≈ 0`(gen 卡整个 think 段闲置 → 零跨卡成立)
- **pipeline**: 两卡均显著 >0(13/15 切分, 逐 token 跨卡)


In [ ]:
_cols = ["think_gpu0_mean", "think_gpu0_max", "think_gpu1_mean", "think_gpu1_max", "think_n"]
_have = [c for c in _cols if c in ok.columns]
gpu_tbl = ok.groupby("placement")[_have].mean().round(1)
print(gpu_tbl.to_string())

_asym_g1 = ok[ok["placement"] == "asym"]["think_gpu1_mean"].mean()
if _asym_g1 < 3:
    print(f"\n✅ asym think 阶段 gen 卡平均利用率 {_asym_g1:.1f}% ≈ 0 — 零跨 PCIe 得到直接证实")
else:
    print(f"\n⚠ asym think 阶段 gen 卡平均利用率 {_asym_g1:.1f}% 明显非零 — "
          f"think 路径可能仍有跨卡调用, 需要 profile 排查")


## 9. 输出一致性: 同 seed 下 asym vs pipeline 的 think 文本

两种放置的数值路径不同(bf16 下计算发生在不同卡/不同切分), **不要求逐字符一致**;
贪心解码下前缀应大体相同, 若很早分叉或文本明显退化才需要警惕。


In [ ]:
_a = pd.read_csv(ASYM_CSV)
_p = pd.read_csv(PIPELINE_CSV)
_m = _a.merge(_p, on=["prompt_idx", "cond_idx", "repeat"], suffixes=("_asym", "_pipe"))


def _common_prefix(s1, s2):
    if not isinstance(s1, str) or not isinstance(s2, str):
        return 0
    n = min(len(s1), len(s2))
    i = 0
    while i < n and s1[i] == s2[i]:
        i += 1
    return i


_m["prefix_chars"] = [_common_prefix(x, y)
                      for x, y in zip(_m["gen_text_asym"], _m["gen_text_pipe"])]
_m["len_asym"] = _m["gen_text_asym"].str.len()
_m["exact_match"] = _m["gen_text_asym"] == _m["gen_text_pipe"]

print(_m.groupby("max_think_token_n_asym")[["exact_match", "prefix_chars", "len_asym"]]
        .agg({"exact_match": "mean", "prefix_chars": "median", "len_asym": "median"})
        .round(2).to_string())

# 抽一条看开头是否合理
_row = _m.iloc[0]
print("\n─── 样例(prompt 0, cap 最小, repeat 0)───")
print("asym :", str(_row["gen_text_asym"])[:300])
print("pipe :", str(_row["gen_text_pipe"])[:300])


## 10. 结论(跑完后填写)

- [ ] pipeline 同机实测基线: ___ tok/s(旧间接基线 18.2)
- [ ] asym: ___ tok/s, 加速比 x___(目标 ≥35 tok/s: 达成/未达成)
- [ ] think 阶段 asym gen 卡利用率 ≈ ___% → 零跨卡是否证实
- [ ] t_image 回归(N=10): asym ___s vs pipeline ___s(x___)
- [ ] 同 seed 文本前缀一致性是否正常

数字确认后同步到 `docs/EXPERIMENT_SUMMARY.md`; 若 asym 仍未达 35 tok/s 且 gpu1≈0,
说明瓶颈在单卡解码本身(kernel launch / launch-bound), 下一步用 `torch.profiler`
看单 token 前向的 GPU 空隙, 评估 CUDA Graph。
